Investigating Data Efficiency and Generalisation in Hybrid Quantum-Classical Vision Transformers

1. Research Overview

This project investigates how the availability of training data affects the classification performance and generalisation of classical and hybrid quantum-classical vision models.

The study will evaluate models using different proportions of the training dataset while maintaining consistent evaluation procedures.

Research Objectives

1. Establish a classical image-classification baseline.
2. Investigate performance at 10%, 25%, 50%, and 100% training-data availability.
3. Evaluate the variability of results across random seeds.
4. Compare predictive performance and computational cost.
5. Determine whether experimental results support any measurable benefit from the hybrid approach.

Experimental Principles

- Use consistent training, validation, and test partitions.
- Control random seeds and document model configurations.
- Use identical evaluation metrics for comparable models.
- Report repeated-run results and variability.
- Avoid assuming quantum advantage before examining experimental evidence.

Initial Dataset

CIFAR-10 will serve as the initial benchmark. It contains colour images from 10 object categories.

Expected Outputs

- Dataset exploration and visualisation.
- Training-data subset definitions.
- Classical baseline results.
- Hybrid model results.
- Comparative plots and statistical summaries.
- Discussion of limitations and conclusions.


2. Environment and Reproducibility

This section imports the libraries required for numerical computation, data analysis, visualisation, and deep learning.

A fixed initial random seed is used to improve reproducibility. However, identical results are not guaranteed across all hardware configurations or nondeterministic operations.

The computation device is selected automatically based on CUDA availability.

In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device:", DEVICE)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

PyTorch: 2.14.1+cu130
Torchvision: 0.29.1+cu130
Device: cpu
NumPy: 2.5.4
Pandas: 3.1.0


3. Dataset Acquisition and Preprocessing

CIFAR-10 is a benchmark dataset containing 60,000 colour images of size 32 × 32 pixels, distributed across 10 classes.

The standard split consists of 50,000 training images and 10,000 test images.

The training set will be used to fit the models. The test set will be reserved for final evaluation and will not be used to select model configurations.

The images will be converted into tensors and normalised using commonly used CIFAR-10 channel statistics.

Reproducibility note: The training data will subsequently be divided into training and validation subsets. Validation data will support model selection, while the test set will remain untouched until final evaluation.

In [ ]:
from torchvision import datasets, transforms
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

train_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),
    train=False,
    download=True,
    transform=transform
)

CLASS_NAMES = train_dataset.classes

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Number of classes:", len(CLASS_NAMES))
print("Class names:", CLASS_NAMES)

5.5%

4. Exploratory Data Visualisation

Visual inspection provides an initial understanding of the image resolution, object categories, and visual diversity present in the dataset.

Ten training images will be displayed with their corresponding class labels.

The normalisation transformation will be reversed for display purposes only. The original dataset tensors will remain unchanged.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]

    # Reverse normalisation for visualisation
    image = image * std + mean
    image = image.clamp(0, 1)
    image = image.permute(1, 2, 0).numpy()

    ax.imshow(image)
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")

plt.suptitle("CIFAR-10 Training Samples", fontsize=16)
plt.tight_layout()
plt.show()

5. Controlled Training-Data Subsets

Objective

Investigate how training-data availability affects image-classification performance.

Four training-data fractions will be evaluated:

- 10%: 5,000 images
- 25%: 12,500 images
- 50%: 25,000 images
- 100%: 50,000 images

These fractions refer to the original CIFAR-10 training set. The test set remains separate and is not used to construct the training subsets.

Methodology

A fixed random seed will be used to select reproducible subsets. The subsets will be nested: the 10% subset will be contained in the 25% subset, which will be contained in the 50% subset, and so on.

This approach makes comparisons across data fractions easier to interpret because larger subsets retain the examples included in smaller subsets.

Important: These are training subsets, not final train/validation splits. We will establish a consistent validation protocol before training models. Any validation examples must be excluded from the data available for model fitting.


In [ ]:
from torch.utils.data import Subset

DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.00]

subset_rng = np.random.default_rng(SEED)

# Generate one reproducible ordering of training indices.
permuted_indices = subset_rng.permutation(len(train_dataset))

train_subsets = {}

for fraction in DATA_FRACTIONS:
    subset_size = int(len(train_dataset) * fraction)
    indices = permuted_indices[:subset_size].tolist()

    train_subsets[fraction] = Subset(train_dataset, indices)

    print(
        f"{fraction:>5.0%} of training data: "
        f"{len(train_subsets[fraction]):,} images"
    )

6. Verify Subset Construction

We verify that the subsets contain the intended number of images and that the nested-subset property holds.

This check helps prevent accidental inconsistencies in the data-efficiency experiment.

In [ ]:
assert len(train_subsets[0.10]) == 5_000
assert len(train_subsets[0.25]) == 12_500
assert len(train_subsets[0.50]) == 25_000
assert len(train_subsets[1.00]) == 50_000

indices_by_fraction = {
    fraction: set(subset.indices)
    for fraction, subset in train_subsets.items()
}

assert indices_by_fraction[0.10].issubset(indices_by_fraction[0.25])
assert indices_by_fraction[0.25].issubset(indices_by_fraction[0.50])
assert indices_by_fraction[0.50].issubset(indices_by_fraction[1.00])

print("All subset-size and nesting checks passed.")

7. Training and Validation Protocol

Objective

Create mutually exclusive training and validation partitions.

The original CIFAR-10 training set contains 50,000 images. We will reserve 5,000 images for validation and use the remaining 45,000 as the training pool.

The four data fractions will now be calculated from this 45,000-image pool. The resulting subset sizes will therefore differ from those calculated using the entire original training set.

The 10,000-image CIFAR-10 test set remains untouched until final evaluation.

Data Leakage Prevention

- Validation images will not be used for gradient updates.
- Test images will not be used for hyperparameter selection.
- The same validation and test partitions will be used when comparing models.
- Subset selection will be reproducible.

In [ ]:
from torch.utils.data import random_split

VALIDATION_SIZE = 5_000
TRAIN_POOL_SIZE = len(train_dataset) - VALIDATION_SIZE

split_generator = torch.Generator().manual_seed(SEED)

train_pool, validation_dataset = random_split(
    train_dataset,
    [TRAIN_POOL_SIZE, VALIDATION_SIZE],
    generator=split_generator
)

print("Training pool:", len(train_pool))
print("Validation set:", len(validation_dataset))
print("Test set:", len(test_dataset))

In [ ]:
pool_indices = np.random.default_rng(SEED).permutation(
    len(train_pool)
)

train_subsets = {}

for fraction in DATA_FRACTIONS:
    subset_size = int(len(train_pool) * fraction)

    indices = pool_indices[:subset_size].tolist()

    train_subsets[fraction] = Subset(
        train_pool,
        indices
    )

    print(
        f"{fraction:>5.0%}: "
        f"{len(train_subsets[fraction]):,} training images"
    )

print("\nValidation images:", len(validation_dataset))
print("Test images:", len(test_dataset))

8. Classical Convolutional Neural Network Baseline

Objective

Establish a classical image-classification baseline for CIFAR-10.

A convolutional neural network (CNN) is used to learn spatial features from images and predict one of the ten CIFAR-10 classes.

Architecture

The baseline contains:

- Two convolutional layers with ReLU activations.
- Max-pooling layers for spatial downsampling.
- A fully connected classification head.
- Ten output logits, one for each class.

Experimental Role

This model provides a reference for later comparison with the hybrid quantum-classical model. The baseline uses standard classical computation and does not contain a quantum circuit.

We will first train on 10% of the training pool. This initial run is a pipeline check, not a final scientific result.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ClassicalCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = ClassicalCNN().to(DEVICE)

print(model)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

9. Mini-batch Data Loading

Training data is processed in mini-batches to manage memory usage and perform stochastic gradient descent.

The initial experiment uses:

- Training fraction: 10%
- Batch size: 128
- Shuffling: enabled for training
- Shuffling: disabled for validation

The validation set remains separate from the training subset.

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

train_loader = DataLoader(
    train_subsets[0.10],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Device:", DEVICE)

10. Training and Evaluation Procedure

The model will be trained using cross-entropy loss and the Adam optimiser.

After each epoch, validation loss and accuracy will be calculated. Validation data will not be used for gradient updates.

We will record:

- Training loss
- Validation loss
- Training accuracy
- Validation accuracy
- Epoch duration

For this initial pipeline check, we will run three epochs. This is not necessarily sufficient for optimal model performance.

In [ ]:
import time


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
    }


def train_model(model, train_loader, validation_loader, epochs=3, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []

    for epoch in range(1, epochs + 1):
        start_time = time.perf_counter()

        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

        train_loss = total_loss / total_samples
        train_accuracy = total_correct / total_samples

        val_metrics = evaluate(
            model,
            validation_loader,
            criterion
        )

        epoch_seconds = time.perf_counter() - start_time

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_loss": val_metrics["loss"],
            "validation_accuracy": val_metrics["accuracy"],
            "epoch_seconds": epoch_seconds,
        }

        history.append(row)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Train acc: {train_accuracy:.2%} | "
            f"Val loss: {val_metrics['loss']:.4f} | "
            f"Val acc: {val_metrics['accuracy']:.2%} | "
            f"Time: {epoch_seconds:.1f}s"
        )

    return pd.DataFrame(history)

11. Initial Baseline Experiment: 10% Training Data

We now train the classical CNN using the 10% training subset.

This experiment checks that:

1. The model's forward and backward passes work.
2. Parameters update during training.
3. Training and validation metrics are computed correctly.
4. Results can be stored for subsequent analysis.

The model has not yet been tuned, and its results should not be interpreted as a definitive benchmark.

In [ ]:
# Reset the seed before initialising the baseline model.
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

baseline_model = ClassicalCNN().to(DEVICE)

baseline_history = train_model(
    model=baseline_model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    epochs=3,
    lr=1e-3
)

baseline_history

12. Baseline Learning Curves

Learning curves show how training and validation performance change over successive epochs.

Training loss should generally decrease as the model learns. Validation performance helps assess whether improvements on the training data translate to unseen examples.

A widening gap between training and validation performance can indicate overfitting, although conclusions should be based on the full training history rather than a single epoch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    baseline_history["epoch"],
    baseline_history["train_loss"],
    marker="o",
    label="Training loss"
)

axes[0].plot(
    baseline_history["epoch"],
    baseline_history["validation_loss"],
    marker="o",
    label="Validation loss"
)

axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    baseline_history["epoch"],
    baseline_history["train_accuracy"],
    marker="o",
    label="Training accuracy"
)

axes[1].plot(
    baseline_history["epoch"],
    baseline_history["validation_accuracy"],
    marker="o",
    label="Validation accuracy"
)

axes[1].set_title("Accuracy by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

13. Initial Classical Baseline Results

Experimental Configuration

- Dataset: CIFAR-10
- Training-data fraction: 10% of the training pool
- Number of training images: 4,500
- Validation images: 5,000
- Model: Classical convolutional neural network
- Optimiser: Adam
- Learning rate: 0.001
- Epochs: 3
- Random seed: 42

Observed Results

Training loss decreased from 2.0910 in the first epoch to 1.5510 in the third epoch. Training accuracy increased from 23.67% to 44.40%.

Validation loss decreased from 1.8294 to 1.4731, while validation accuracy increased from 36.70% to 48.20%.

Interpretation

The decreasing training and validation losses indicate that the model is learning useful patterns from the training data. The improvement in validation accuracy suggests that these learned patterns generalise to the held-out validation set.

No clear evidence of overfitting is visible in this short training run. However, longer training and repeated experiments are required to characterise generalisation reliably.

Limitations

This is an initial pipeline-validation experiment, not a fully tuned baseline. The model was trained for only three epochs and evaluated using one random seed.

Further experiments will be required before comparing classical and hybrid quantum-classical architectures.

14. Data-Efficiency Experiment: Varying Training-Set Size

Objective

Investigate how the amount of training data affects the classification performance of a classical convolutional neural network (CNN) on CIFAR-10.

Experimental Design

The same CNN architecture will be trained using four different fractions of the training pool:

- 10%: 4,500 images
- 25%: 11,250 images
- 50%: 22,500 images
- 100%: 45,000 images

A fixed validation set of 5,000 images will be used to evaluate each experiment. The model architecture, optimiser, learning rate, and number of epochs will remain consistent.

Each experiment will start with the same model-initialisation seed. Training-set shuffling will also be controlled for reproducibility.

Evaluation Metrics

- Training loss
- Training accuracy
- Validation loss
- Validation accuracy
- Training time per epoch

The validation results will help determine whether additional training data improves classification performance. The test set will remain untouched until the final evaluation stage.

Important: This initial experiment uses one random seed per training fraction. The results are preliminary; repeated runs with multiple seeds will be needed to assess variability.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader

# Experiment configuration
EPOCHS = 5
LEARNING_RATE = 1e-3
BATCH_SIZE = 128

# Create directory for experiment results
os.makedirs("results", exist_ok=True)

all_histories = []
models_by_fraction = {}

for fraction in DATA_FRACTIONS:

    print("\n" + "=" * 60)
    print(f"Training fraction: {fraction:.0%}")
    print(f"Training images: {len(train_subsets[fraction]):,}")
    print("=" * 60)

    # Reset random seeds
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    # Reproducible data shuffling
    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)

    fraction_train_loader = DataLoader(
        train_subsets[fraction],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=loader_generator
    )

    # Initialise a fresh CNN
    model = ClassicalCNN().to(DEVICE)

    # Train and validate
    history = train_model(
        model,
        fraction_train_loader,
        validation_loader,
        epochs=EPOCHS,
        lr=LEARNING_RATE
    )

    # Record experiment configuration
    history["training_fraction"] = fraction
    history["training_images"] = len(train_subsets[fraction])

    all_histories.append(history)
    models_by_fraction[fraction] = model

    # Save individual experiment history
    history.to_csv(
        f"results/cnn_history_{int(fraction * 100)}pct.csv",
        index=False
    )

# Combine all experiment histories
experiment_history = pd.concat(
    all_histories,
    ignore_index=True
)

experiment_history.to_csv(
    "results/cnn_data_efficiency_history.csv",
    index=False
)

print("\nAll experiments completed.")

display(experiment_history)

15. Comparative Analysis of Data Fractions

This section compares the final performance of the classical CNN trained with different amounts of CIFAR-10 training data.

The comparison includes training accuracy, validation accuracy, training loss, validation loss, and total training time.

The difference between training and validation accuracy provides an indication of the generalisation gap. These results will help determine whether increasing training-data quantity improves classification performance.

In [ ]:
# Extract the final epoch for each training fraction
final_results = (
    experiment_history
    .sort_values("epoch")
    .groupby("training_fraction")
    .tail(1)
    .sort_values("training_fraction")
    .reset_index(drop=True)
)

# Calculate total training time across all epochs
total_times = (
    experiment_history
    .groupby("training_fraction")["epoch_seconds"]
    .sum()
    .rename("total_training_seconds")
)

final_results = final_results.merge(
    total_times,
    on="training_fraction"
)

# Calculate generalisation gap in percentage points
final_results["generalisation_gap"] = (
    final_results["train_accuracy"]
    - final_results["validation_accuracy"]
) * 100

comparison = final_results[
    [
        "training_fraction",
        "training_images",
        "train_accuracy",
        "validation_accuracy",
        "generalisation_gap",
        "train_loss",
        "validation_loss",
        "total_training_seconds"
    ]
].copy()

comparison["train_accuracy"] *= 100
comparison["validation_accuracy"] *= 100

comparison = comparison.rename(columns={
    "training_fraction": "Training Fraction",
    "training_images": "Training Images",
    "train_accuracy": "Train Accuracy (%)",
    "validation_accuracy": "Validation Accuracy (%)",
    "generalisation_gap": "Generalisation Gap (pp)",
    "train_loss": "Train Loss",
    "validation_loss": "Validation Loss",
    "total_training_seconds": "Training Time (s)"
})

comparison = comparison.round(3)

comparison.to_csv(
    "results/cnn_data_efficiency_comparison.csv",
    index=False
)

display(comparison)

16. Visualising Data Efficiency

Visualisations provide an intuitive comparison of model performance and computational cost across training-set sizes.

The first figure plots validation accuracy against the percentage of training data used. The second compares training time across experiments.

These figures will help identify whether additional data improves generalisation and how the computational cost changes with dataset size.

In [ ]:
import matplotlib.pyplot as plt

plot_results = final_results.sort_values("training_fraction")

plt.figure(figsize=(8, 5))

plt.plot(
    plot_results["training_fraction"] * 100,
    plot_results["validation_accuracy"] * 100,
    marker="o"
)

plt.xlabel("Training Data Used (%)")
plt.ylabel("Validation Accuracy (%)")
plt.title("CNN Validation Accuracy vs. Training-Set Size")
plt.xticks([10, 25, 50, 100])
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    "results/cnn_validation_accuracy_vs_data.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    plot_results["training_fraction"] * 100,
    plot_results["total_training_seconds"],
    marker="o"
)

plt.xlabel("Training Data Used (%)")
plt.ylabel("Total Training Time (seconds)")
plt.title("CNN Training Time vs. Training-Set Size")
plt.xticks([10, 25, 50, 100])
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(
    "results/cnn_training_time_vs_data.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

17. Learning-Curve Analysis

Learning curves show how training and validation performance change over successive epochs.

Training loss should generally decrease as the model learns. Validation loss and accuracy help assess whether these improvements transfer to unseen data.

Comparing learning curves across training fractions can reveal whether smaller datasets learn more slowly, reach lower performance, or exhibit a larger generalisation gap.

The curves will be interpreted alongside the final metrics rather than used as standalone evidence of generalisation.

In [ ]:
fractions = sorted(experiment_history["training_fraction"].unique())

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    sharex=True
)

axes = axes.flatten()

for ax, fraction in zip(axes, fractions):
    subset = (
        experiment_history[
            experiment_history["training_fraction"] == fraction
        ]
        .sort_values("epoch")
    )

    ax.plot(
        subset["epoch"],
        subset["train_loss"],
        marker="o",
        label="Training Loss"
    )

    ax.plot(
        subset["epoch"],
        subset["validation_loss"],
        marker="s",
        label="Validation Loss"
    )

    ax.set_title(f"{fraction:.0%} of Training Data")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("Training and Validation Loss Across Data Fractions")
fig.tight_layout()

fig.savefig(
    "results/cnn_learning_curves_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
    sharex=True
)

axes = axes.flatten()

for ax, fraction in zip(axes, fractions):
    subset = (
        experiment_history[
            experiment_history["training_fraction"] == fraction
        ]
        .sort_values("epoch")
    )

    ax.plot(
        subset["epoch"],
        subset["train_accuracy"] * 100,
        marker="o",
        label="Training Accuracy"
    )

    ax.plot(
        subset["epoch"],
        subset["validation_accuracy"] * 100,
        marker="s",
        label="Validation Accuracy"
    )

    ax.set_title(f"{fraction:.0%} of Training Data")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy (%)")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("Training and Validation Accuracy Across Data Fractions")
fig.tight_layout()

fig.savefig(
    "results/cnn_learning_curves_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

18. Saving the Classical CNN Baseline

The trained CNN models and experiment configuration will be saved to support reproducibility and future comparisons.

The model trained on the full training pool will serve as the principal classical baseline for subsequent experiments.

The saved checkpoint will contain the model parameters, architecture identifier, training configuration, random seed, and final validation metrics.

This checkpoint will support later evaluation and comparison with the hybrid quantum-classical model.

In [ ]:
import json

# Select the model trained with 100% of the training pool
full_data_fraction = 1.0
full_data_model = models_by_fraction[full_data_fraction]

full_data_metrics = (
    final_results[
        final_results["training_fraction"] == full_data_fraction
    ]
    .iloc[0]
)

# Save model checkpoint
checkpoint = {
    "model_state_dict": full_data_model.state_dict(),
    "model_name": "ClassicalCNN",
    "dataset": "CIFAR-10",
    "training_fraction": full_data_fraction,
    "training_images": len(train_subsets[full_data_fraction]),
    "validation_images": len(validation_dataset),
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "validation_accuracy": float(
        full_data_metrics["validation_accuracy"]
    ),
    "validation_loss": float(
        full_data_metrics["validation_loss"]
    )
}

torch.save(
    checkpoint,
    "results/classical_cnn_full_data.pt"
)

# Save experiment configuration
config = {
    "dataset": "CIFAR-10",
    "model": "ClassicalCNN",
    "seed": SEED,
    "data_fractions": DATA_FRACTIONS,
    "training_pool_size": len(train_pool),
    "validation_size": len(validation_dataset),
    "test_size": len(test_dataset),
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE)
}

with open(
    "results/experiment_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(config, file, indent=4)

print("Saved full-data classical CNN checkpoint.")
print("Saved experiment configuration.")
print(
    "Full-data validation accuracy:",
    f"{full_data_metrics['validation_accuracy']:.2%}"
)